In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingWarmRestarts, SequentialLR
from torch.utils.data import Dataset, DataLoader
import math
import scanpy as sc
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
import time as pytime
import argparse
import os
from scipy.stats import pearsonr
from pathlib import Path

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [3]:
adata = sc.read('./Code/downstream_analysis_code/ExploratoryAnalysis/Data/a549_166drugMoA.h5ad')
gptEmbed_df = pd.read_csv('./Data/FinalData/gptEmbed_Jul9_final.csv',index_col=0)
MFP_df = pd.read_csv("./Data/FinalData/compounds_512MFP_wholeDat_fixed.csv",index_col=0)
drug_targets = pd.read_csv("./Data/FinalData/compounds_target_multihot_full.csv", index_col=0)

In [4]:
import math
import numpy as np
import torch
from torch.utils.data import Dataset
from scipy import sparse

class GenePerturbationDataset(Dataset):
    """
    Returns tensors for:
      - baseline_expr (978,)
      - cell_stats     (1956,)  # per-cell mean || variance
      - drug_gptEmbed  (512,)
      - drug_MFP       (512,)
      - drug_targets   (1164,)
      - [y_true        (978,)]   # ONLY in mode='train'
      - dose_feat      (1,)
      - time_feat      (1,)

    Key changes for inference:
      * mode='infer' → no y_true; baseline taken from the current row (adata_new).
      * robust to missing split columns and missing 'control' column.
    """

    def __init__(
        self,
        adata,
        gptEmbed_df,
        MFP_df,
        drug_targets,
        split_strategy: str = "cell_split",
        split_value: str = "train",
        mode: str = "train",
    ):
        assert mode in {"train", "infer"}, "mode must be 'train' or 'infer'"
        self.mode         = mode
        self.adata        = adata
        self.gptEmbed_df  = gptEmbed_df
        self.MFP_df       = MFP_df
        self.drug_targets = drug_targets

        # ---- choose indices
        if (split_strategy in adata.obs.columns) and (mode == "train"):
            self.indices = np.where(adata.obs[split_strategy] == split_value)[0]
        else:
            # inference (or split column not present): use all rows
            self.indices = np.arange(adata.n_obs)

        # ---- pre-compute per-cell mean & variance for cell_stats
        obs = adata.obs
        X   = adata.X
        if sparse.issparse(X):
            X = X.tocsr()

        # prefer controls if available; otherwise use all rows
        if "control" in obs.columns and (obs["control"] == 1).any():
            ctrl_mask = (obs["control"] == 1).to_numpy()
            X_ctrl = X[ctrl_mask]
            if sparse.issparse(X_ctrl):
                X_ctrl = X_ctrl.toarray()
            cell_ids = obs.loc[ctrl_mask, "cell_id"].to_numpy()
        else:
            # fallback: use everything (works for adata_new which holds baselines)
            if sparse.issparse(X):
                X_ctrl = X.toarray()
            else:
                X_ctrl = np.asarray(X)
            cell_ids = obs["cell_id"].to_numpy()

        self.cell_stats = {}
        for cid in np.unique(cell_ids):
            mat = X_ctrl[cell_ids == cid]
            mu  = mat.mean(0, dtype=np.float32)
            var = mat.var(0, dtype=np.float32)
            self.cell_stats[cid] = torch.from_numpy(np.concatenate([mu, var]))  # (1956,)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = int(self.indices[idx])
        obs_row  = self.adata.obs.iloc[real_idx]

        # --- baseline expression
        if self.mode == "train":
            # original behavior: use paired control as baseline
            paired_id  = obs_row["paired_control_index"]
            paired_idx = self.adata.obs.index.get_loc(paired_id)
            x_baseline = self.adata.X[paired_idx]
        else:
            # inference: baseline is the current row in adata_new (already a chosen control)
            x_baseline = self.adata.X[real_idx]

        # dense 1D
        if sparse.issparse(x_baseline):
            x_baseline = x_baseline.toarray().ravel().astype("float32")
        else:
            x_baseline = np.asarray(x_baseline).ravel().astype("float32")

        # --- cell stats by cell_id (A549 in your adata_new)
        cell_id    = obs_row["cell_id"]
        cell_stats = self.cell_stats[cell_id]  # (1956,)

        # --- drug features (looked up by pert_iname)
        drug_name = obs_row["pert_iname"]
        try:
            x_drug_gptEmbed = self.gptEmbed_df.loc[drug_name].to_numpy(dtype="float32")
        except KeyError:
            raise KeyError(f"gptEmbed_df missing drug '{drug_name}'")
        try:
            x_drug_MFP = self.MFP_df.loc[drug_name].to_numpy(dtype="float32")
        except KeyError:
            raise KeyError(f"MFP_df missing drug '{drug_name}'")
        try:
            x_drug_targets = self.drug_targets.loc[drug_name].to_numpy(dtype="float32")
        except KeyError:
            raise KeyError(f"drug_targets missing drug '{drug_name}'")

        # --- dose/time features (same transform as training)
        dose = float(obs_row["dose"])        # you've renamed to 'dose'
        time = float(obs_row["pert_time"])   # hours
        dose_feat = torch.tensor(math.log10(dose + 1.0), dtype=torch.float32)
        time_feat = torch.tensor(math.log10(time + 1.0), dtype=torch.float32)

        if self.mode == "train":
            # y_true = perturbed expression at real_idx
            y_true = self.adata.X[real_idx]
            if sparse.issparse(y_true):
                y_true = y_true.toarray().ravel().astype("float32")
            else:
                y_true = np.asarray(y_true).ravel().astype("float32")

            return (
                torch.from_numpy(x_baseline),        # (978,)
                cell_stats,                           # (1956,)
                torch.from_numpy(x_drug_gptEmbed),    # (512,)
                torch.from_numpy(x_drug_MFP),         # (512,)
                torch.from_numpy(x_drug_targets),     # (1164,)
                torch.from_numpy(y_true),             # (978,)
                dose_feat,                            # scalar
                time_feat,                            # scalar
            )
        else:
            # inference: no target
            return (
                torch.from_numpy(x_baseline),        # (978,)
                cell_stats,                           # (1956,)
                torch.from_numpy(x_drug_gptEmbed),    # (512,)
                torch.from_numpy(x_drug_MFP),         # (512,)
                torch.from_numpy(x_drug_targets),     # (1164,)
                dose_feat,                            # scalar
                time_feat,                            # scalar
            )


In [5]:
infer_ds = GenePerturbationDataset(
    adata=adata,                 # built with random A549 controls
    gptEmbed_df=gptEmbed_df,         # indexed by pert_iname
    MFP_df=MFP_df,                   # indexed by pert_iname
    drug_targets=drug_targets,    # indexed by pert_iname
    split_strategy="cell_split",     # ignored in infer if missing
    split_value="train",             # ignored in infer
    mode="infer",                    # <<< key change
)


# Create DataLoader
infer_loader = DataLoader(infer_ds, batch_size=128, shuffle=False)

In [6]:
# ────────────────────────────────────────────────────────────────────────────
# Utility: lightweight denoising auto‑encoder
# ────────────────────────────────────────────────────────────────────────────
class DenoisingAE(nn.Module):
    """Two-layer symmetric auto-encoder with configurable bottleneck."""
    def __init__(self, in_dim: int, latent_dim: int, dropout: float = 0.1):
        super().__init__()
        h_dim = max(latent_dim * 2, in_dim // 2)
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, h_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(h_dim, latent_dim), nn.GELU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, h_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(h_dim, in_dim)
        )
    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return z, x_hat


# ╭─────────────────────────────────────────────────────────────╮
# │ 1.  Encoder‑only projectors                                 │
# ╰─────────────────────────────────────────────────────────────╯
class Enc2Layer(nn.Module):
    """
    Two‑layer MLP encoder (same shape as the old AE encoder).
    512 → h_dim → latent_dim
    """
    def __init__(self, in_dim: int = 512,
                       latent_dim: int = 128,
                       dropout: float = 0.1):
        super().__init__()
        h_dim = max(latent_dim * 2, in_dim // 2)      # mirror AE logic
        self.net = nn.Sequential(
            nn.Linear(in_dim, h_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(h_dim, latent_dim),
            nn.GELU()
        )

    def forward(self, x):           # (B, in_dim) → (B, latent_dim)
        return self.net(x)


class Enc3Layer(nn.Module):
    """
    Three‑layer funnel: 512 → 256 → 128 → latent_dim.
    BatchNorm stabilises deeper stack; second dropout is lighter.
    """
    def __init__(self, in_dim: int = 512,
                       latent_dim: int = 128,
                       dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.GELU(), nn.BatchNorm1d(256),
            nn.Dropout(dropout),
            nn.Linear(256, 128),    nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(128, latent_dim)                 # no activation = linear code
        )

    def forward(self, x):           # (B, in_dim) → (B, latent_dim)
        return self.net(x)

In [7]:
'''
Model Transformer encoder for gene
'''
class GenePerturbationTransformer(nn.Module):
    def __init__(self,
                 d_model: int = 32,
                 num_heads: int = 8,
                 num_encoder_layers: int = 2,
                 dropout: float = 0.2,
                 max_len: int = 978,
                 d_hidden: int = 64,
                 n_genes_target: int = 1164,
                 fp_bits: int = 512,
                 llm_dim: int = 512,
                 ae_latent: int = 128):
        super().__init__()
        self.d_model   = d_model
        self.num_heads = num_heads
        self.max_len   = max_len

        # ───── gene‑specific **two‑layer** MLP  (3 → d_mid → d_model)  ----------
        d_mid = d_model // 2  # e.g. 16 when d_model=32
        # first layer weights/bias  (S, 3, d_mid)
        self.W1_gene = nn.Parameter(torch.empty(max_len, 3, d_mid))
        self.b1_gene = nn.Parameter(torch.empty(max_len, d_mid))
        # second layer  (S, d_mid, d_model)
        self.W2_gene = nn.Parameter(torch.empty(max_len, d_mid, d_model))
        self.b2_gene = nn.Parameter(torch.empty(max_len, d_model))
        nn.init.xavier_uniform_(self.W1_gene)
        nn.init.zeros_(self.b1_gene)
        nn.init.xavier_uniform_(self.W2_gene)
        nn.init.zeros_(self.b2_gene)

        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=num_heads, dropout=dropout)
        self.gene_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)

        # ───────────── drug-feature projections & fusion  ───────────────────
        # ───────────── modality‑specific AEs ───────────────────────────────
        # 1) LLM‑derived embedding (512 → 256 → 128)
        self.ae_llm = Enc3Layer(llm_dim, latent_dim=ae_latent, dropout=dropout)

        # mute drug target
        # 2) Drug‑target binary vector: first linear proj → 512, then 512 → 256 → 128
        # self.tgt_proj = nn.Linear(n_genes_target, llm_dim)
        # self.ae_tgt   = Enc3Layer(llm_dim, latent_dim=ae_latent, dropout=dropout)

        # 3) Morgan fingerprint bits (512 → 256 → 128)
        self.ae_fp  = Enc3Layer(fp_bits, latent_dim=ae_latent, dropout=dropout)
        
        
        def mlp_proj(in_dim, out_dim):
            return nn.Sequential(
                nn.Linear(in_dim, out_dim)
            )
        
        self.llm_proj_k = mlp_proj(ae_latent, num_heads*d_model)
        self.llm_proj_v = mlp_proj(ae_latent, num_heads*d_model)

        # mute drug target
        # self.tgt_proj_k = mlp_proj(ae_latent, num_heads*d_model)
        # self.tgt_proj_v = mlp_proj(ae_latent, num_heads*d_model)

        self.fp_proj_k  = mlp_proj(ae_latent, num_heads*d_model)
        self.fp_proj_v  = mlp_proj(ae_latent, num_heads*d_model)

        # three cross-attention blocks
        self.xattn1 = nn.MultiheadAttention(d_model, num_heads,
                                            dropout)
        self.xattn2 = nn.MultiheadAttention(d_model, num_heads,
                                            dropout)
        self.xattn3 = nn.MultiheadAttention(d_model, num_heads,
                                            dropout)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        
        self.dropout = nn.Dropout(dropout)

        # ───────────── shared FFN & per-gene head (unchanged) ──────────────
        self.shared_ffn = nn.Sequential(
            nn.Linear(d_model, d_hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_hidden, d_model)
        )
        
        # ───────────── NEW: gene‑specific FiLM parameters ──────────────────
        self.gamma_gene = nn.Parameter(torch.ones(max_len, d_model))  # (S,d)
        self.beta_gene  = nn.Parameter(torch.zeros(max_len, d_model)) # (S,d)
        
        # variance-driven FiLM parameters (cell-specific)
        self.var_gamma = nn.Parameter(torch.zeros(max_len, d_model))
        self.var_beta  = nn.Parameter(torch.zeros(max_len, d_model))
        
        self.W = nn.Parameter(torch.empty(max_len, d_model))
        self.b = nn.Parameter(torch.empty(max_len))
        
        self.Wmlp1 = nn.Parameter(torch.empty(max_len, d_model, d_mid))
        self.bmlp1 = nn.Parameter(torch.zeros(max_len, d_mid))
        self.Wmlp2 = nn.Parameter(torch.empty(max_len, d_mid))    # to scalar
        self.bmlp2 = nn.Parameter(torch.zeros(max_len))

        for p in (self.W, self.Wmlp1, self.Wmlp2):
            nn.init.xavier_uniform_(p)
            
        for p in (self.b, self.bmlp1, self.bmlp2):
            nn.init.zeros_(p)
            
        def _make_gate():
            g = nn.Sequential(
                    nn.Linear(2, 16), nn.GELU(),
                    nn.Linear(16, 1),
                    nn.Softplus()        # ≥ 0, smooth
            )
            with torch.no_grad():       # start near zero gain
                g[-2].weight.zero_()
                g[-2].bias.fill_(-4.0)  # Softplus(-4) ≈ 0.018
            return g

        self.gate_attn1 = _make_gate()   # for LLM cross‑attention
        # mute drug target
        # self.gate_attn2 = _make_gate()   # for target cross‑attention
        self.gate_attn3 = _make_gate()   # for FP   cross‑attention
            

    # ----------------------------------------------------------------------
    def _kv(self, proj_k, proj_v, feat):
        """
        proj_* : Linear to (H·D)
        feat   : (B, feat_dim)
        returns k,v with shape (B, H, D)   (seq_len = H tokens)
        """
        B = feat.size(0)
        # (B, H * d_model)  →  reshape →  (H, B, d_model)
        k = proj_k(feat) \
              .view(B, self.num_heads, self.d_model) \
              .permute(1, 0, 2)
        v = proj_v(feat) \
              .view(B, self.num_heads, self.d_model) \
              .permute(1, 0, 2)
        return k, v

    # ----------------------------------------------------------------------
    def forward(self,
                x_base: torch.Tensor,   # (B,S)
                cell_stats: torch.Tensor, # (B,2S)  = [μ ‖ σ²]
                x_llm: torch.Tensor,
                x_fp: torch.Tensor,
                x_tgt: torch.Tensor,
                dose_feat: torch.Tensor, 
                time_feat: torch.Tensor):
        B, S = x_base.shape
        assert S == self.max_len, "gene dim mismatch"

        # prepare 3‑vector per gene ------------------------------------
        mu  = cell_stats[:, :S]
        var = cell_stats[:, S:]
        # concat input features
        x_feat = torch.stack([x_base, mu, var], dim=-1)        # (B,S,3)
    
        # 0) denoise modalities --------------------------------------------
        z_llm = self.ae_llm(x_llm)                       # (B,512)

        # Mute drug target
        # tgt_emb = self.tgt_proj(x_tgt)                           # (B,512)
        # z_tgt = self.ae_tgt(tgt_emb)                    # (B,512)

        z_fp = self.ae_fp(x_fp)                        # (B,512)
        
        # 0.1) prepare the dose and time features
        scalar_pair = torch.stack([dose_feat, time_feat], dim=-1)   # (B,2)
        
        
        # 1) encode genes ----------------------------------------------------
        # --- per‑gene 2‑layer MLP projection ----------------------------------
        h1 = torch.einsum('bsi,sid->bsd', x_feat, self.W1_gene) + self.b1_gene
        h1 = torch.nn.functional.gelu(h1)
        x  = torch.einsum('bsd,sdk->bsk', h1, self.W2_gene) + self.b2_gene
        
        x = x.transpose(0,1)                                    # (S,B,d)
        x = self.gene_encoder(x)

        # 2) genes ⟵ LLM  ----------------------------------------------------
        k1,v1 = self._kv(self.llm_proj_k, self.llm_proj_v, z_llm)
        out,_ = self.xattn1(query=x, key=k1, value=v1)
        
        g1 = self.gate_attn1(scalar_pair).unsqueeze(0)  # (1,B,1)
        x = self.norm1(x + self.dropout(g1 * out))

        # mute drug target
        # 3) genes ⟵ targets  -----------------------------------------------
        # k2,v2     = self._kv(self.tgt_proj_k, self.tgt_proj_v, z_tgt)
        # out,_     = self.xattn2(query=x, key=k2, value=v2)
        
        # g2 = self.gate_attn2(scalar_pair).unsqueeze(0)
        # x = self.norm2(x + self.dropout(g2 * out))

        # 4) genes ⟵ fingerprints  ------------------------------------------
        k3,v3 = self._kv(self.fp_proj_k, self.fp_proj_v, z_fp)
        out,_ = self.xattn3(query=x, key=k3, value=v3)
        
        g3 = self.gate_attn3(scalar_pair).unsqueeze(0)
        x = self.norm3(x + self.dropout(g3 * out))
        
        x = x.transpose(0, 1)                                  # (B,S,d) (back to batch‑first)

        # 4) FFN + per-gene head -------------------------------------------
        h      = self.shared_ffn(x)
        
        # 5) variance-driven FiLM
        var_unsq = var.unsqueeze(-1)  # (B,S,1)
        gamma_v  = 1 + var_unsq * self.var_gamma.unsqueeze(0)
        beta_v   = var_unsq * self.var_beta.unsqueeze(0)
        h = gamma_v * h + beta_v
        
        # 6) Gene‑specific FiLM adaptation ---------------------------------
        h = self.gamma_gene.unsqueeze(0) * h + self.beta_gene.unsqueeze(0)  # (B,S,d)
        
        
        # linear path (unchanged)
        linear_out = torch.einsum('bsd,sd->bs', h, self.W) + self.b      # (B,S) → (B,)

        # residual MLP path
        h_m = torch.einsum('bsd,sdm->bsm', h, self.Wmlp1) + self.bmlp1   # (B,S,d_mid)
        h_m = torch.nn.functional.gelu(h_m)
        mlp_out = torch.einsum('bsm,sm->bs', h_m, self.Wmlp2) + self.bmlp2

        logits = linear_out + mlp_out
        
        return logits

In [8]:
model = GenePerturbationTransformer(d_model=32, num_heads=8, num_encoder_layers=4, dropout=0.1, max_len=978, d_hidden=64, 
                                    n_genes_target=1225, fp_bits=512, llm_dim=512, ae_latent=128)
model = model.to(device)

model_dir = Path("./Model")
ckpt_name = f"transformer_d32h8l4_cell_split5_dp1_MSECor_lr1_CosSche_3XAttn_sepGene2EncPred_newAttn_FiLM_Enc3DimReducLa128_CellAware_frontEndMLPsimp_whole_DoseTimeAsScalarXattns_LLMMFP_first50epoch.pth"
ckpt_path = model_dir / ckpt_name

# optional: check it exists
if not ckpt_path.exists():
    raise FileNotFoundError(f"Checkpoint not found: {ckpt_path}")

checkpoint = torch.load(ckpt_path, map_location=device)

print(checkpoint["epoch"])
model.load_state_dict(checkpoint['model_state'] 
                       if 'model_state' in checkpoint else checkpoint)

85


/nas/longleaf/home/meisheng/.local/lib/python3.11/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


<All keys matched successfully>

In [9]:
def pred_model(model, dataloader, device):
    """
    Evaluate gene-specific metrics on the validation set.
    
    For each gene (across all samples):
      - Computes the squared errors and calculates their mean (MSE Mean)
        and standard deviation (MSE Std).
      - Computes a single R² score over all samples.
      - Computes the Pearson correlation coefficient over all samples.
    
    Args:
        model (torch.nn.Module): The trained model.
        dataloader (torch.utils.data.DataLoader): Validation DataLoader.
        device (torch.device): The device (CPU/GPU).
    
    Returns:
        mse_df (pd.DataFrame): DataFrame with columns ['Gene', 'MSE Mean', 'MSE Std'],
                               sorted in ascending order by 'MSE Mean'.
        r2_df (pd.DataFrame): DataFrame with columns ['Gene', 'R2'],
                              sorted in descending order by 'R2'.
        pearson_df (pd.DataFrame): DataFrame with columns ['Gene', 'Pearson'],
                                   sorted in descending order by 'Pearson'.
    """
    # Set the model to evaluation mode.
    model.eval()
    all_baseline = []
    all_y_pred = []
    all_pred_delta = []
    

    # Collect predictions and ground truth values.
    with torch.no_grad():
        for x_base, cell_stats, x_llm, x_fp, x_tgt, dose, time in dataloader:  # ← correct order
            x_base = x_base.to(device)
            cell_stats = cell_stats.to(device)
            x_llm  = x_llm.to(device)
            x_tgt  = x_tgt.to(device)
            x_fp   = x_fp.to(device)
            dose   = dose.to(device)
            time   = time.to(device)

            y_pred = model(x_base, cell_stats, x_llm, x_fp, x_tgt, dose, time)
            
            delta_pred = y_pred - x_base
            
            all_baseline.append(x_base.cpu())
            all_y_pred.append(y_pred.cpu())
            
            all_pred_delta.append(delta_pred.cpu())


    # --- concatenate the batch lists ---
    baseline_mat = torch.cat(all_baseline).numpy()   # shape: (n_samples, n_genes)
    pred_mat     = torch.cat(all_y_pred).numpy()     # same shape
    all_delta_pred_mat = torch.cat(all_pred_delta).numpy()

    # optional: keep track of which AnnData row each sample came from
    row_order = dataloader.dataset.indices           # numpy array of adata row positions

    # --- build the DataFrame ---
    df_preds = pd.DataFrame({
        'adata_row' : row_order,          # or adata.obs_names[row_order] for IDs
        'Baseline'  : list(baseline_mat), # store each vector as a single cell (dtype=object)
        'Pred'      : list(pred_mat),
        'Pred_delta': list(all_delta_pred_mat)
    })

    return df_preds

In [10]:
pred_df = pred_model(model, infer_loader, device)

In [11]:
pred_df.shape

(1110, 4)

In [12]:
pred_df

,adata_row,Baseline,Pred,Pred_delta
0,0,"[5.9033637, 3.7364464, 6.5990195, 10.812334, 1...","[5.9792356, 4.10353, 6.8142586, 10.595537, 10....","[0.075871944, 0.36708355, 0.21523905, -0.21679..."
1,1,"[5.938199, 4.481189, 7.6138487, 10.841192, 11....","[6.2113304, 4.524112, 6.9415073, 11.1485, 10.7...","[0.27313137, 0.04292345, -0.67234135, 0.307308..."
2,2,"[5.518738, 4.1447062, 6.2350345, 10.367332, 10...","[5.8112736, 4.3455667, 6.7123947, 10.293549, 1...","[0.29253578, 0.2008605, 0.47736025, -0.0737838..."
3,3,"[5.814579, 4.624229, 6.516044, 10.152318, 10.4...","[5.907591, 4.6312466, 6.8488855, 10.054441, 10...","[0.093011856, 0.0070176125, 0.3328414, -0.0978..."
4,4,"[5.518738, 4.1447062, 6.2350345, 10.367332, 10...","[5.8112836, 4.345568, 6.712449, 10.293485, 10....","[0.2925458, 0.20086193, 0.4774146, -0.07384777..."
...,...,...,...,...
1105,1105,"[5.5731497, 4.1467695, 6.2118683, 10.379757, 1...","[6.0251055, 4.411783, 6.672214, 10.330835, 10....","[0.4519558, 0.2650137, 0.46034575, -0.04892158..."
1106,1106,"[5.7799897, 4.578484, 6.958024, 10.125143, 10....","[6.145192, 4.5898046, 6.9741616, 10.178596, 10...","[0.36520243, 0.011320591, 0.0161376, 0.0534524..."
1107,1107,"[5.713047, 4.2773013, 7.724593, 9.735673, 10.7...","[6.19503, 4.396104, 7.280571, 10.123165, 10.60...","[0.48198318, 0.11880255, -0.44402218, 0.387492..."
1108,1108,"[5.771897, 4.0092163, 6.5993557, 10.4324, 10.3...","[6.1418753, 4.319522, 6.725429, 10.419043, 10....","[0.36997843, 0.3103056, 0.12607336, -0.0133571..."


In [13]:
results_dir = f'./Code/downstream_analysis_code/ExploratoryAnalysis/Data'
out_dir = Path(results_dir)
out_dir.mkdir(parents=True, exist_ok=True)   # create folders if missing
out_csv = out_dir / f"pred_df_A549_166drugsMoA_infer.csv"
pred_df.to_csv(out_csv, index=True, header=True)
print(f"Saved to {out_csv}")

Saved to /work/users/m/e/meisheng/Dissertation/Experiments_Feb152025/code/transformer/Jul7WholeData/inference/A549/pred_df_A549_166drugsMoA_infer.csv
